<br>
<a href="https://github.com/aperture-systems-lab">
    <img src="assets/banner_semillero.png" width="955" style="margin: 0px 0px 12px;"/>
</a>
<h1 style="line-height: 1.4;"><font color="#29c4d9"><b>Cómo funcionan las redes neuronales</b></font></h1>
<h2><b>Notebook 2: </b>Clasificación</h2>

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

import utils

torch.manual_seed(42)

----

<br>

## **Parte 0:** Nuestro objetivo

<div style="float: right; width: 46%; min-width: 220px; max-width: 420px; margin: 4px 50px 12px 30px;">
  <img src="assets/titanic.png" width="100%" alt="Hundimiento del Titanic"
       style="display: block; transform: rotate(-1.5deg);
              filter: drop-shadow(0px 12px 20px rgba(41, 196, 217, 0.35));"/>
</div>

Todos vimos *Titanic*, ¿sí o qué? Por si acaso, un recorderis pa los de peor memoria: 

La taquillera película de James Cameron (2.200 millones de dólares de recaudó) cuenta la historia de Jack Dawson (Leonardo Dicaprio), un joven pobre que viaja en tercera clase, y Rose DeWitt Bukater, una joven de primera clase comprometida con un millonario. Que terminan en una epopeya romantica en un viaje transatlantico.

Ambos el 15 de abril de 1912, durante su viaje inaugural, el RMS Titanic (considerado "insumergible") se hundió tras chocar con un iceberg. No había botes salvavidas para todos, y de las 2224 personas a bordo murieron 1502.

Al analizar los datos del evento vimos que el pobre muere más fácil que el rico y que se le daba prioridad a las mujeres y niños. Esto sugiere que dado un choque existen gente más probable en morir que otra, en el final casi estaba escrito: Rose era mujer y de primera clase mientras que Jack era hombre y de tercera clase.

El objetivo de este notebook es construir un modelo de **redes neuronales** que prediga **si un pasajero sobrevivió o no** a partir de sus datos.

<div style="clear: both;"></div>

----

<br>

## **Parte 1:** Los datos

Usaremos el registro de pasajeros del Titanic, el reto clásico de [Kaggle](https://www.kaggle.com/c/titanic), repartido en `train.csv` y `test.csv`.

A diferencia del notebook anterior, acá la salida no es un número cualquiera sino una decisión entre dos opciones.

Por lo que queremos un modelo de la forma:

$$
F: x \rightarrow y \in \{0, 1\}
$$

que a partir de la información de un pasajero estime qué tan probable era que sobreviviera. 

A este tipo de problema se le llama **clasificación**. Mientras que el problema del notebook anterior se la llamaba **regresión**.

In [ ]:
train_df = pd.read_csv("data/titanic/train.csv")
test_df = pd.read_csv("data/titanic/test.csv")

pasajeros_raw = pd.concat([train_df, test_df], ignore_index=True)

print("dimensiones:", pasajeros_raw.shape)
pasajeros_raw.head()

----

<br>

## **Parte 2:** Preparación de los datos

In [ ]:
def preprocess(df):
    df = df.copy()

    # Quita signos de puntuación del inicio y final de cada palabra del nombre
    def normalize_name(x):
        return " ".join([v.strip(",()[].\"'") for v in x.split(" ")])

    # Devuelve la última parte del ticket (normalmente el número)
    def ticket_number(x):
        return x.split(" ")[-1]

    # Devuelve el prefijo del ticket unido con "_", o "NONE" si no tiene
    def ticket_item(x):
        items = x.split(" ")
        if len(items) == 1:
            return "NONE"
        return "_".join(items[0:-1])

    # Aplica la limpieza
    df["Name"] = df["Name"].apply(normalize_name)
    df["Ticket_number"] = df["Ticket"].apply(ticket_number)
    df["Ticket_item"] = df["Ticket"].apply(ticket_item)
    return df


preprocessed_df = preprocess(train_df)

# Selecionar las variables de entrada
input_features = list(preprocessed_df.columns)
input_features.remove("Ticket")
input_features.remove("PassengerId")
input_features.remove("Survived")
input_features.remove("Name")

print(f"Input features: {input_features}")
preprocessed_df.head(5)

In [ ]:
preprocessed_train_df, preprocessed_serving_df = train_test_split(
    preprocessed_df,
    test_size=0.2,
    random_state=42
)

print(f"{len(preprocessed_train_df)} ejemplos de entrenamiento, {len(preprocessed_serving_df)} ejemplos de prueba.")

In [ ]:
def to_tensors(df, input_features):
    df = df[input_features + ["Survived"]].copy()

    # Rellena valores faltantes
    df = df.fillna({
        "Age": df["Age"].median(), 
        "Fare": df["Fare"].median(),
        "Embarked": "S", 
        "Cabin": "NONE"})

    # Convierte cada columna de texto en números enteros
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = pd.factorize(df[col])[0]

    X = torch.tensor(df[input_features].values, dtype=torch.float32)
    y = torch.tensor(df["Survived"].values, dtype=torch.float32).unsqueeze(1)

    return X, y


X_train, y_train = to_tensors(preprocessed_train_df, input_features)
X_val, y_val = to_tensors(preprocessed_serving_df, input_features)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32, shuffle=False)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

----

<br>

## **Parte 3:** La red neuronal

Red neuronal para clasifacación

In [ ]:
class RedNeuronal(nn.Module):

    def __init__(self, n_features, neuronas):
        super().__init__()
        self.layer_1 = nn.Linear(n_features, neuronas)
        self.relu = nn.ReLU()
        self.layer_2 = nn.Linear(neuronas, neuronas)
        self.layer_3 = nn.Linear(neuronas, 1)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.relu(x)
        x = self.layer_3(x)
        return x

Como ahora es clasificación, cambiamos `MSELoss` por `BCEWithLogitsLoss`.

In [ ]:
modelo = RedNeuronal(X_train.shape[1], neuronas=64)
funcion_perdida = nn.BCEWithLogitsLoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.01)

historial_red = []

for epoca in range(1000):
    optimizador.zero_grad()                            # 1. borrar los gradientes viejos
    predicciones = modelo(X_train)                     # 2. predecir
    perdida = funcion_perdida(predicciones, y_train)   # 3. medir el error
    perdida.backward()                                 # 4. calcular gradientes
    optimizador.step()                                 # 5. actualizar parametros

    historial_red.append(perdida.item())

    if (epoca + 1) % 200 == 0:
        print(f"Época {epoca + 1} — pérdida: {perdida}")

# Graficando la perdida
utils.plot_loss(historial_red)

----

<br>

## **Parte 4:** ¿Qué queremos maximizar?

### Accuracy

Porcentaje de predicciones correctas hechas por el modelo

$$
Accuracy = \frac{Predicciones\ correctas}{Todas\ las\ predicciones}
$$

- **Maximizar**
- Puede ser poco confiable con datos desbalanceados

De todos los pasajeros, ¿en cuántos acertó el modelo, ya sea que sobrevivieron o no?

In [ ]:
def accuracy(y_real, y_pred):
    ### Inicio del código
    correctas = torch.sum(y_pred == y_real)
    total = y_real.shape[0]

    output = correctas / total

    ### fin del código
    return output

### Confusion Matriz

![https://cdn-images-1.medium.com/max/679/1*YXQWFPQP7lzLBgv9K0hCPg.png](assets/embarazo.png)

<br>

No es una métrica, es la tabla de la que salen todas las demás.

<br>

|  | **Dijo: murió** | **Dijo: sobrevivió** |
|---|---|---|
| **Murió** | Verdadero negativo | Falso positivo |
| **Sobrevivió** | Falso negativo | Verdadero positivo |

- Filas: lo que pasó
- Columnas: lo que dijo el modelo

<br>

**Aciertos**

- **Verdadero negativo** — Jack murió y el modelo dijo que moría
- **Verdadero positivo** — Rose sobrevivió y el modelo dijo que sobrevivía

**Errores**

- **Falso positivo** — Jack murió, pero el modelo lo dio por salvado
- **Falso negativo** — Rose sobrevivió, pero el modelo la dio por muerta

In [ ]:
def matriz_confusion(y_real, y_pred):
    ### Inicio del código


    ### fin del código
    return output

### Precision 

Mide qué tan seguido son correctas las predicciones positivas del modelo

$$
Precision = \frac{Verdaderos\ positivos}{Verdaderos\ positivos + Falsos\ positivos}
$$

- **Maximizar**
- Usar cuando los falsos positivos son costosos

De los pasajeros que el modelo dijo que sobrevivieron, ¿cuántos sobrevivieron realmente?

In [ ]:
def precision(y_real, y_pred):
    ### Inicio del código



    ### fin del código
    return output

### Recall

Mide qué tanto identifica el modelo los verdaderos positivos entre todos los positivos reales

$$
Recall = \frac{Verdaderos\ positivos}{Verdaderos\ positivos + Falsos\ negativos}
$$

- **Maximizar**
- Usar cuando los falsos negativos son costosos

De todos los pasajeros que realmente sobrevivieron, ¿a cuántos logró detectar el modelo?


In [ ]:
def recall(y_real, y_pred):
    ### Inicio del código


    ### fin del código
    return output

### F1 Score

Equilibra los falsos positivos y los falsos negativos

$$
F1\ Score = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
$$

- **Maximizar**
- Preferible a la accuracy con datos desbalanceados

¿Qué tan bien identifica el modelo a los sobrevivientes, sin equivocarse demasiado al decir que alguien sobrevivió ni dejando escapar a demasiados que sí lo hicieron?

In [ ]:
def f1_score(y_real, y_pred):
    ### Inicio del código


    ### fin del código
    return output

Para simplificar usaremos esta función que nos va calcular las metricas de una pasada 

In [ ]:
def evaluar_metricas(modelo, dataloader, umbral=0.5):
    modelo.eval()  
    
    reales = []
    predichas = []

    with torch.no_grad():
        for X, y in dataloader:
            probabilidades = torch.sigmoid(modelo(X))     
            predichas.append((probabilidades >= umbral).float())
            reales.append(y)

    y_real = torch.cat(reales)
    y_pred = torch.cat(predichas)

    return {
        "accuracy": accuracy(y_real, y_pred).item(),
        "precision": precision(y_real, y_pred).item(),
        "recall": recall(y_real, y_pred).item(),
        "f1": f1_score(y_real, y_pred).item(),
        "matrix": matriz_confusion(y_real, y_pred),
    }

In [ ]:
resultados_red = evaluar_metricas(modelo, val_loader)

utils.plot_results(resultados_red, title="Red neuronal — conjunto de validación")

----

<br>

## **Parte 5:** Búsqueda de hiperparámetros con Optuna

Los **hiperparámetros** los decidimos antes de entrenar. En vez de adivinarlos, [Optuna](https://optuna.org/) los busca:

| Hiperparámetro | Valores |
|---|---|
| **Neuronas** | 8, 16, 32, 64, 128 |
| **Capas ocultas** | 1, 2, 3 |
| **Activación** | ReLU, Tanh |
| **Dropout** | 0, 0.2, 0.4 |
| **Tasa de aprendizaje** | 0.001, 0.003, 0.01, 0.03 |
| **Épocas** | 250, 500, 1000 |

Son 1.080 combinaciones, Optuna prueba 50 y nos quedamos con la de mejor F1.

In [ ]:
class RedConfigurable(nn.Module):

    def __init__(self, n_features, neuronas, capas, activacion, dropout):
        super().__init__()
        activaciones = {"relu": nn.ReLU, "tanh": nn.Tanh}

        bloques = []
        entradas = n_features
        for _ in range(capas):
            bloques += [nn.Linear(entradas, neuronas), activaciones[activacion](), nn.Dropout(dropout)]
            entradas = neuronas
        bloques.append(nn.Linear(entradas, 1))

        self.red = nn.Sequential(*bloques)

    def forward(self, x):
        return self.red(x)


def entrenar_red(neuronas, capas, activacion, dropout, tasa, epocas):
    modelo = RedConfigurable(X_train.shape[1], neuronas, capas, activacion, dropout)
    funcion_perdida = nn.BCEWithLogitsLoss()
    optimizador = optim.Adam(modelo.parameters(), lr=tasa)

    for epoca in range(epocas):
        optimizador.zero_grad()                            # 1. borrar los gradientes viejos
        predicciones = modelo(X_train)                     # 2. predecir
        perdida = funcion_perdida(predicciones, y_train)   # 3. medir el error
        perdida.backward()                                 # 4. calcular gradientes
        optimizador.step()                                 # 5. actualizar parametros

    return modelo


def objetivo(trial):
    neuronas = trial.suggest_categorical("neuronas", [8, 16, 32, 64, 128])
    capas = trial.suggest_categorical("capas", [1, 2, 3])
    activacion = trial.suggest_categorical("activacion", ["relu", "tanh"])
    dropout = trial.suggest_categorical("dropout", [0.0, 0.2, 0.4])
    tasa = trial.suggest_categorical("tasa", [0.001, 0.003, 0.01, 0.03])
    epocas = trial.suggest_categorical("epocas", [250, 500, 1000])

    modelo = entrenar_red(neuronas, capas, activacion, dropout, tasa, epocas)
    return evaluar_metricas(modelo, val_loader)["f1"]


estudio = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
estudio.optimize(objetivo, n_trials=50)

print("Mejor combinación:", estudio.best_params)
print("Mejor F1:", estudio.best_value)

In [ ]:
mejor_red = entrenar_red(**estudio.best_params)

resultados_red = evaluar_metricas(mejor_red, val_loader)

utils.plot_results(resultados_red, title="Mejor red neuronal — conjunto de validación")

----

<br>

## **Parte 6:** ¿Hacía falta una red?

Probemos algo mucho más sencillo: una regresión logística, una sola capa lineal sin capas ocultas.

$$
\sigma(w \cdot x + b) = \frac{1}{1 + e^{-(w \cdot x + b)}}
$$

In [ ]:
class RegresionLogistica(nn.Module):
    def __init__(self, n_features, media, desviacion):
        super().__init__()
        self.media = media
        self.desviacion = desviacion
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        x = (x - self.media) / self.desviacion   # SGD necesita las entradas en la misma escala
        return self.linear(x)


modelo = RegresionLogistica(X_train.shape[1], X_train.mean(dim=0), X_train.std(dim=0))
funcion_perdida = nn.BCEWithLogitsLoss()
optimizador = optim.SGD(modelo.parameters(), lr=0.1)

historial_log = []

for epoca in range(2000):
    optimizador.zero_grad()                            # 1. borrar los gradientes viejos
    predicciones = modelo(X_train)                     # 2. predecir
    perdida = funcion_perdida(predicciones, y_train)   # 3. medir el error
    perdida.backward()                                 # 4. calcular gradientes
    optimizador.step()                                 # 5. actualizar parametros

    historial_log.append(perdida.item())

    if (epoca + 1) % 200 == 0:
        print(f"Época {epoca + 1} — pérdida: {perdida}")


utils.plot_loss(historial_log)

resultados_log = evaluar_metricas(modelo, val_loader)

utils.plot_results(resultados_log, title="Regresión logística — conjunto de validación")

### Comparativa

In [ ]:
utils.plot_comparison(
    {
        "red neuronal (Optuna)": resultados_red,
        "regresión logística": resultados_log,
    },
    title="Conjunto de validación",
)

print(f"red neuronal:        {sum(p.numel() for p in mejor_red.parameters()):>4} parámetros, {len(estudio.trials)} entrenamientos")
print(f"regresión logística: {sum(p.numel() for p in modelo.parameters()):>4} parámetros, 1 entrenamiento")

### Recorderis

Antes de entrenar, para tener en cuenta los errores más comunes con redes neuronales según Andrej Karpathy.

<div style="text-align: center; margin: 12px 0px 16px;">
  <img src="assets/karpathy.png" width="800" alt="Tweet de Andrej Karpathy sobre los errores más comunes con redes neuronales"
       style="filter: drop-shadow(0px 12px 20px rgba(41, 196, 217, 0.35));"/>
  <br>
  <a href="https://x.com/karpathy/status/1013244313327681536?lang=es">https://x.com/karpathy/status/1013244313327681536?lang=es</a>
</div>

-----

<br>

### **Siguiente notebook:**

[`03_hotdog_or_not.ipynb`](03_hotdog_or_not.ipynb)

### <font color="#29c4d9">**Notebook 2 listo.**</font>

<br>

---

<div style="margin-top: 50px;"><center><a href="https://github.com/aperture-systems-lab"><img src="assets/banner_logo.png" width="955"/></a></center></div>